---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — include notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 🔀 Partial Least Squares (PLS)
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> PCR finds components that explain X. PLS finds components that explain both X **and** Y simultaneously. When p is large and Y is hard to predict, PLS often outperforms PCR.

### What you'll learn
- How PLS components differ from PCA components
- NIPALS algorithm — the iterative deflation procedure
- Choosing the number of PLS components M via cross-validation
- Head-to-head comparison: OLS vs Ridge vs Lasso vs PCR vs PLS

### Time: ~40 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Hitters dataset (same as PCR, Ridge, Lasso, Subset Selection) ────────
try:
    hitters = pd.read_csv('https://www.statlearning.com/s/Hitters.csv')
    print(f'✓ Hitters.csv (ISLP): {hitters.shape}')
except Exception:
    hitters = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Hitters.csv')
    print(f'✓ Hitters.csv (fallback): {hitters.shape}')

hitters = hitters.dropna()
hitters_enc = pd.get_dummies(hitters, columns=['League','Division','NewLeague'],
                              drop_first=True, dtype=float)
hitters_enc['log_salary'] = np.log(hitters_enc['Salary'])
hitters_enc = hitters_enc.drop(columns=['Salary'])

target   = 'log_salary'
features = [c for c in hitters_enc.columns if c != target]
X = hitters_enc[features].values.astype(float)
y = hitters_enc[target].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler  = StandardScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_te_s  = scaler.transform(X_te)
p = len(features)

print(f'  Features ({p}): {features}')
print(f'  Train: {X_tr_s.shape}  Test: {X_te_s.shape}')
print(f'  Python {sys.version.split()[0]} | sklearn {__import__("sklearn").__version__}')
print('✓ Setup complete')


## 📐 Part 1 — PCR vs PLS: The Core Difference

Both methods reduce X to M components before regressing. The difference is **how** the components are built:

| | Component direction maximises… |
|---|---|
| **PCA / PCR** | Variance in **X** only |
| **PLS** | Covariance between **X and Y** |

**Consequence:** PCA can spend its first component on a direction in X that barely relates to Y. PLS always points toward Y.

**Algorithm (NIPALS — Nonlinear Iterative Partial Least Squares):**
1. Find weight vector **w** that maximises cov(Xw, Y)
2. Compute scores **t = Xw** (the PLS component)
3. Regress Y on t → loading **q**
4. Deflate: remove the variance explained by t from both X and Y
5. Repeat for the next component

In [ ]:
# ── Visualise the PCR vs PLS component difference ────────────────────────────
pca  = PCA(n_components=2).fit(X_tr_s)
pls2 = PLSRegression(n_components=2).fit(X_tr_s, y_tr)

# Project data onto first 2 components of each method
Z_pca = pca.transform(X_tr_s)
Z_pls = pls2.transform(X_tr_s)[0]   # PLSRegression returns (X_scores, Y_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, Z, title, method in zip(
        axes,
        [Z_pca, Z_pls],
        ['PCA (PCR) — components explain X variance',
         'PLS — components explain X→Y covariance'],
        ['PCA', 'PLS']):
    sc = ax.scatter(Z[:,0], Z[:,1], c=y_tr, cmap='RdYlGn',
                    alpha=0.6, s=25, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='log(Salary)')
    ax.set_xlabel(f'{method} Component 1')
    ax.set_ylabel(f'{method} Component 2')
    ax.set_title(title)

plt.suptitle('PCR vs PLS: How well do the first 2 components separate salary levels?\n'
             'PLS components are explicitly oriented toward Y', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# Correlation of first component with Y
r_pca = np.corrcoef(Z_pca[:,0], y_tr)[0,1]
r_pls = np.corrcoef(Z_pls[:,0], y_tr)[0,1]
print(f"Correlation of Component 1 with log(Salary):")
print(f"  PCA  Component 1: r = {r_pca:.3f}")
print(f"  PLS  Component 1: r = {r_pls:.3f}")
print()
print("📌 PLS Component 1 is more correlated with Y — it was built to be.")
print("   PCA Component 1 maximises variance in X regardless of Y.")


## 🔬 Part 2 — Choosing M via Cross-Validation

Like PCR, PLS has a single hyperparameter M (number of components). We choose it by cross-validation.

Unlike PCR where M = p gives OLS exactly, PLS with M = p does **not** reproduce OLS — PLS deflates the residuals differently.

In [ ]:
# ── PLS: cross-validate over M = 1 … min(p, n_tr) ───────────────────────────
max_M    = min(p, X_tr_s.shape[0] - 1)
M_range  = range(1, max_M + 1)
cv_mses  = []

for M in M_range:
    pls = PLSRegression(n_components=M)
    mse = -cross_val_score(pls, X_tr_s, y_tr, cv=10,
                            scoring='neg_mean_squared_error').mean()
    cv_mses.append(mse)

best_M_pls   = list(M_range)[np.argmin(cv_mses)]
best_mse_pls = min(cv_mses)

# OLS baseline
ols_cv_mse = -cross_val_score(LinearRegression(), X_tr_s, y_tr, cv=10,
                               scoring='neg_mean_squared_error').mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(list(M_range), cv_mses, 'o-', color='#e85d20', lw=2,
        markersize=4, label='PLS 10-fold CV MSE')
ax.axhline(ols_cv_mse, color='grey', lw=1.5, ls='--',
           label=f'OLS baseline CV MSE = {ols_cv_mse:.4f}')
ax.axvline(best_M_pls, color='#1a7a45', lw=2, ls='--',
           label=f'Best M = {best_M_pls}  (CV MSE = {best_mse_pls:.4f})')
ax.set_xlabel('Number of PLS Components M')
ax.set_ylabel('10-fold CV MSE (log salary)')
ax.set_title('PLS: Choosing the Number of Components via Cross-Validation')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

pls_best = PLSRegression(n_components=best_M_pls).fit(X_tr_s, y_tr)
pls_test_mse = mean_squared_error(y_te, pls_best.predict(X_te_s))

print(f"Best M = {best_M_pls} PLS components")
print(f"PLS test MSE: {pls_test_mse:.4f}  RMSE: {pls_test_mse**0.5:.4f}")


## 📊 Part 3 — The Grand Comparison: All Methods on Hitters

Now we compare every shrinkage/dimensionality method covered in this chapter on the **same** train/test split of Hitters. This is the key table from ISLP Chapter 6.

In [ ]:
# ── Full comparison: OLS, Ridge, Lasso, PCR, PLS ─────────────────────────────
from sklearn.decomposition import PCA

# Refit all models on the training set
ols   = LinearRegression().fit(X_tr_s, y_tr)
ridge = RidgeCV(alphas=np.logspace(-3, 4, 100), cv=10).fit(X_tr_s, y_tr)
lasso = LassoCV(alphas=np.logspace(-4, 2, 100), cv=10, random_state=42,
                max_iter=5000).fit(X_tr_s, y_tr)

# PCR — find best M
pcr_mses = []
for M in range(1, p+1):
    pipe = Pipeline([('pca', PCA(n_components=M)), ('lr', LinearRegression())])
    mse  = -cross_val_score(pipe, X_tr_s, y_tr, cv=10,
                             scoring='neg_mean_squared_error').mean()
    pcr_mses.append(mse)
best_M_pcr = int(np.argmin(pcr_mses)) + 1
pcr_best   = Pipeline([('pca', PCA(n_components=best_M_pcr)),
                        ('lr', LinearRegression())]).fit(X_tr_s, y_tr)

# Compute test MSE and CV MSE for all
models = {
    'OLS (all features)':      ols,
    f'Ridge (α={ridge.alpha_:.3f})': ridge,
    f'Lasso ({np.sum(lasso.coef_!=0)} features)': lasso,
    f'PCR (M={best_M_pcr})':  pcr_best,
    f'PLS (M={best_M_pls})':  pls_best,
}

print(f"{'Method':30s}  {'Test MSE':>10}  {'Test RMSE':>10}  {'Test RMSE ($)':>14}")
print("-" * 72)
comparison = {}
for name, model in models.items():
    y_pred      = model.predict(X_te_s).ravel()
    test_mse    = mean_squared_error(y_te, y_pred)
    test_rmse   = test_mse ** 0.5
    rmse_dollar = mean_squared_error(np.exp(y_te), np.exp(y_pred))**0.5
    comparison[name] = test_mse
    print(f"{name:30s}  {test_mse:>10.4f}  {test_rmse:>10.4f}  ${rmse_dollar:>12,.0f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names = list(comparison.keys())
mses  = list(comparison.values())
colors = ['#888','#1e5fa8','#e85d20','#1a7a45','#c0392b']
bars = axes[0].barh(names, mses, color=colors, edgecolor='white', height=0.6)
axes[0].axvline(min(mses), color='gold', lw=2, ls='--', label='Best')
axes[0].set_xlabel('Test MSE (log salary)')
axes[0].set_title('Test MSE Comparison — Hitters Dataset')
axes[0].legend()
for bar, v in zip(bars, mses):
    axes[0].text(v + 0.0005, bar.get_y()+bar.get_height()/2,
                 f'{v:.4f}', va='center', fontsize=9)

# Coefficient paths: how many features each method uses
n_nonzero = {
    'OLS':   p,
    'Ridge': p,
    'Lasso': int(np.sum(lasso.coef_ != 0)),
    'PCR':   best_M_pcr,
    'PLS':   best_M_pls,
}
axes[1].bar(n_nonzero.keys(), n_nonzero.values(),
            color=['#888','#1e5fa8','#e85d20','#1a7a45','#c0392b'],
            edgecolor='white')
axes[1].axhline(p, color='grey', lw=1, ls='--', label=f'All {p} features')
axes[1].set_ylabel('Active features / components')
axes[1].set_title('Model Complexity — Features or Components Used')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

best_method = min(comparison, key=comparison.get)
print(f"\n📌 Best method on this split: {best_method}")
print()
print("Key takeaways:")
print("  • OLS with all features often overfits — especially with correlated predictors")
print("  • Ridge keeps all features but shrinks; Lasso performs selection")
print("  • PCR compresses X before regressing — helps with multicollinearity")
print("  • PLS uses fewer components than PCR and often achieves similar performance")
print("  • No method universally wins — results depend on the dataset")
print("    One split is not enough to declare a winner; use CV for model selection")


## ✅ Part 4 — PLS Loadings and Coefficient Interpretation

Unlike PCR, PLS loadings directly reflect how each original feature contributes to predicting Y. The x-loadings show which features drive each component; the y-loadings show how each component relates to Y.

In [ ]:
# ── PLS loadings — which features matter for each component ─────────────────
x_loadings = pd.DataFrame(
    pls_best.x_loadings_,
    index=features,
    columns=[f'Comp {i+1}' for i in range(best_M_pls)]
)

print(f"PLS X-loadings (M={best_M_pls} components):")
print(f"{'Feature':20s}", end='')
for c in x_loadings.columns: print(f"  {c:>10}", end='')
print()
print("-" * (22 + 12 * best_M_pls))
for fname, row in x_loadings.iterrows():
    print(f"{fname:20s}", end='')
    for v in row:
        marker = ' ★' if abs(v) == row.abs().max() else ''
        print(f"  {v:>10.4f}{marker[:2]:2s}", end='')
    print()

print()
print("★ = largest absolute loading for that component")
print()

# VIP scores — Variable Importance in Projection
# VIP > 1 indicates a feature important for the PLS model
W  = pls_best.x_weights_          # shape (p, M)
T  = pls_best.x_scores_           # shape (n, M)
Q  = pls_best.y_loadings_         # shape (1, M)

ss = (T**2).sum(axis=0) * (Q**2).sum(axis=0)  # SS explained per component
vip_scores = np.sqrt(p * (W**2 @ ss) / ss.sum())

vip_df = pd.DataFrame({'Feature': features, 'VIP': vip_scores})           .sort_values('VIP', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors_vip = ['#1a7a45' if v >= 1.0 else '#1e5fa8' if v >= 0.8 else '#aaa'
              for v in vip_df['VIP']]
ax.barh(vip_df['Feature'], vip_df['VIP'], color=colors_vip, edgecolor='white')
ax.axvline(1.0, color='#e85d20', lw=2, ls='--', label='VIP = 1 (important threshold)')
ax.axvline(0.8, color='orange',  lw=1, ls=':', label='VIP = 0.8 (moderate)')
ax.set_xlabel('VIP Score')
ax.set_title(f'Variable Importance in Projection — PLS (M={best_M_pls})')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

important = vip_df[vip_df['VIP'] >= 1.0]['Feature'].tolist()
print(f"Features with VIP ≥ 1.0 (important for PLS model): {important}")
print()
print("📌 VIP > 1: feature contributes more than average to all PLS components.")
print("   Use VIP scores for feature selection in a PLS workflow.")


## 💼 Exercise

**Task 1 — PLS for classification:** `PLSRegression` can be used for binary classification by treating the 0/1 target as continuous, then thresholding predictions at 0.5. Using the Default dataset, build a PLS classifier with M=1,2,3 and compare test accuracy to Logistic Regression.

**Task 2 — Scree comparison:** On one chart, plot the cumulative variance explained by PCA components vs the cumulative R² explained by PLS components (using `pls_best.score()` per component). Which method reaches 80% faster?

**Task 3 — Sensitivity to M:** Rerun the grand comparison but with M fixed at 1, 5, and 10 for both PCR and PLS. How sensitive is each method to the choice of M?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables available:
#   X_tr_s, X_te_s, y_tr, y_te, features, p
#   pls_best  — fitted PLSRegression(n_components=best_M_pls)
#   best_M_pls, best_M_pcr, pcr_best

# Task 1 — PLS as a binary classifier (use Default dataset)
# from sklearn.datasets import ... (load Default separately)

# Task 2 — PLS vs PCA cumulative R²
pca_full = PCA().fit(X_tr_s)
print("Cumulative variance explained:")
print(f"{'M':>4}  {'PCA cumvar':>12}  {'PLS R² (approx)':>16}")
print("-" * 36)
for M in range(1, min(11, p+1)):
    pca_var = pca_full.explained_variance_ratio_[:M].sum()
    pls_m   = PLSRegression(n_components=M).fit(X_tr_s, y_tr)
    pls_r2  = pls_m.score(X_tr_s, y_tr)
    print(f"{M:>4}  {pca_var:>12.3f}  {pls_r2:>16.3f}")

# Task 3 — Sensitivity to M
print("\nSensitivity to M:")
print(f"{'M':>4}  {'PCR test MSE':>14}  {'PLS test MSE':>14}")
print("-" * 36)
for M in [1, 5, 10]:
    if M <= p:
        pcr_m = Pipeline([('pca', PCA(n_components=M)),
                           ('lr',  LinearRegression())]).fit(X_tr_s, y_tr)
        pls_m = PLSRegression(n_components=M).fit(X_tr_s, y_tr)
        pcr_mse = mean_squared_error(y_te, pcr_m.predict(X_te_s))
        pls_mse = mean_squared_error(y_te, pls_m.predict(X_te_s).ravel())
        print(f"{M:>4}  {pcr_mse:>14.4f}  {pls_mse:>14.4f}")


In [ ]:
# @title 📝 Quiz — Partial Least Squares
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** What does PLS optimise when constructing components, and how does this differ from PCA?
# @markdown **Q2:** Why might PLS use fewer components than PCR to achieve the same predictive accuracy?
# @markdown **Q3:** What is a VIP score and what threshold indicates an important feature?
# @markdown **Q4:** Name one advantage and one disadvantage of PLS compared to Lasso.
# @markdown **Q5:** If PLS(M=1) outperforms OLS, what does this suggest about the Hitters data?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Partial Least Squares"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong/partial, say which Part to revisit")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*